# Cross-entity attention pretrain on Colab

Free T4-runtime path while we wait for GCE GPU quota approval. The three tarballs live in `gs://orbit-wars-shipping/` (already uploaded by `scripts/pack_for_gpu.sh`); this notebook authenticates to GCP, pulls them, untars, runs the pretrain, and writes the trained checkpoint back to GCS so we can `gsutil cp` it locally.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. The Google account you sign into Colab with must own (or have read/write on) the GCP project `analog-receiver-489214-e9` and bucket `gs://orbit-wars-shipping/`.

Total runtime: ~3 min setup + ~10 min training = ~15 min.

## 1. Verify GPU

Free Colab sometimes hands out CPU runtimes silently. Abort early if so — no point training a 30-epoch run on a CPU runtime when we have a CPU at home.

In [ ]:
import torch, sys
if not torch.cuda.is_available():
    sys.exit('No GPU runtime — Runtime → Change runtime type → T4 GPU, then re-run.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

## 2. Authenticate to GCP & pull tarballs

`google.colab.auth.authenticate_user()` makes the credentials of your Google account available to `gsutil` — no service-account keys to manage. The tarballs are about 130 MB total.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT = 'analog-receiver-489214-e9'
BUCKET = 'gs://orbit-wars-shipping'

!gcloud config set project {PROJECT}

In [ ]:
import os
WORK = '/content/orbit-wars'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

for name in ('code.tgz', 'data.tgz', 'weights.tgz'):
    !gsutil cp {BUCKET}/{name} .

## 3. Unpack + install

In [ ]:
%cd {WORK}
!tar xzf code.tgz
!tar xzf data.tgz
!tar xzf weights.tgz

# Top-level layout sanity check
!ls -la

In [ ]:
%cd {WORK}
# Most of requirements.txt is already on Colab. Install only the
# specifically-required ones if any are missing — torch is preinstalled
# at the runtime version, so don't reinstall it.
!pip install -q -r requirements.txt --no-deps
# kaggle-environments is the heaviest external dep; install if missing
# (the agent registry imports it at module load).
!pip install -q kaggle-environments

## 4. Sanity-check the imports

Catches any version-skew between the local PyTorch we packed against and Colab's preinstalled version before we burn 10 min of GPU time on an import error.

In [ ]:
import sys
sys.path.insert(0, WORK)

from agents.transformer_v1.pretrain.cross_entity import (
    CrossEntitySnapshotDataset, CrossEntityPretrainModel, train,
    train_gradual_unfreeze,
)
from agents.transformer_v1.paths import (
    FLEET_RUNS_DIR, PLANET_RUNS_DIR, ENTITY_RUNS_DIR, CROSS_ENTITY_RUNS_DIR,
)
from pathlib import Path

for d in (FLEET_RUNS_DIR, PLANET_RUNS_DIR, ENTITY_RUNS_DIR, CROSS_ENTITY_RUNS_DIR):
    runs = sorted(p for p in Path(d).iterdir() if p.is_dir())
    print(f'{d}:')
    for r in runs:
        print(f'  {r.name}')
print('imports OK')

## 5. Train

Resume from the latest `cross_entity_best.pt` packed into `weights.tgz`, then run **Stage 1** of gradual unfreezing only. That keeps the already-trained cross-entity transformer + heads fixed as the starting point and thaws just the `PlanetEntityEncoder` for a short, low-LR follow-up run. Outputs land in `data/runs/cross_entity/<timestamp>/`.

In [ ]:
%cd {WORK}
!python -m agents.transformer_v1.pretrain.cross_entity \
    --train-mode gradual-unfreeze \
    --stage-epochs 5 \
    --batch-size 64 --num-workers 2 \
    --eval-every 1 --device cuda

## 6. Push results back to GCS

Tar the run dir (best/last ckpts + log + test summary) and upload back to the same bucket so we can `gsutil cp` from local. Trivially small (~1 MB).

In [ ]:
import time
from pathlib import Path

runs = sorted(p for p in Path(WORK, 'data', 'runs', 'cross_entity').iterdir() if p.is_dir())
if not runs:
    raise SystemExit('no run dir found under data/runs/cross_entity/')
latest = runs[-1]
print(f'pushing {latest.relative_to(WORK)} ...')

stamp = time.strftime('%Y%m%d-%H%M%S')
tar_name = f'cross_entity_run_{stamp}.tgz'
%cd {latest.parent}
!tar czf /tmp/{tar_name} {latest.name}
!gsutil cp /tmp/{tar_name} {BUCKET}/{tar_name}
print(f'\nPull from local with:\n  gsutil cp {BUCKET}/{tar_name} .')

## 7. Quick test-set summary

Read the JSON the pretrain script writes and print a clean per-head table. Use this to sanity-check before running `gsutil cp` locally.

In [ ]:
import json
summary = json.loads((latest / 'test_summary.json').read_text())
print(f'{"head":<32s}  {"loss":>8s}  {"acc":>6s}')
for name, m in summary.items():
    acc = f'{m["acc"]:.3f}' if 'acc' in m else '-'
    print(f'{name:<32s}  {m["loss"]:>8.4f}  {acc:>6s}')